# Will it be an early Spring?

On February 2<sup>nd</sup> every year Punxsutawney Phil makes a prediction about if there will be an early Spring or if Winter will continue for 6 more weeks (till about mid-March). He is however not very accurate (well, according to [The Inner Circle](https://www.groundhog.org/inner-circle) he is 100% correct but the human handler may not interpret his response correctly). The overall goal is to be able to predict if it will be an early Spring.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help). To schedule an appointment, look at the client's Google Calendar and then reach out to them to propose a time. I estimate the first meeting will need about 20 minutes. Come prepared.

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
   * Must provide a function `generate_samples(years: list[int], phil_preds: list[bool]) -> pd.DataFrame` that takes a list of years, a list of `True`/`False`s for Phil's prediction for those years, and gives back a data frame for those years matching the formatting of the train and test sets.
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes a the train or test set (or anything returned by `generate_samples()`) and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preperation should be noted here but actually implemented in the Preperation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit all data files and a pickled final model along with this notebook.

# Frame the Problem and Look at the Big Picture

## 1. Define the objective in business terms

A farm in western Pennsylvania can plant a newly developed, fast-growing corn variety. In an early spring the farm can fit in two harvests instead of one, which means significantly more profit. The catch:

* The seeds must be ordered by March 1, are expensive (≈ 1/5 of the revenue of the extra harvest), expire quickly, and are highly regulated (so they presumably can't be stored for next year or resold).
* If the farm buys seeds and spring is not early, the seed money is wasted.
* If the farm doesn't buy seeds and spring is early, the extra harvest's profit is missed.

**Business objective:** every year, using only information available before March 1, decide whether to buy the seeds so that the farm's long-run profit is maximized.

### What "early spring" means for this client 

It is an early spring if the seeds can be planted on or before May 1 and the first 10 days after planting satisfy all of:

| Condition | Threshold |
|---|---|
| Average soil temperature (top 3 inches) over the 10 days | ≥ 50 °F (otherwise the seeds won't sprout) |
| Minimum air temperature | never below 30 °F |
| Average air temperature | ≥ 40 °F |
| Snow on the ground | none, on any of the 10 days |

After those 10 days, cold weather no longer matters. Soil temperature is the most critical factor.

## 2. How will your solution be used?

* **When:** once per year, in the last days of February, right before the March 1 order deadline.
* **Input:** weather and soil data up to the end of February, climate indices published by then (El Niño / Arctic Oscillation), and Punxsutawney Phil's Feb 2 call.
* **Output:** a probability of an early spring and a buy / don't-buy recommendation (the threshold is chosen from the cost structure, see step 5).
* **Who uses it:** the farm's decision maker. It is a decision-support tool, the farmer makes the final call.
* **Mode:** offline / batch. One prediction per year. The model can be retrained each May once that year's outcome is known.

## 3. What are the current solutions/workarounds?

* **Punxsutawney Phil.** The client considers this more of a joke. Phil predicts an early spring only rarely (~20 of ~130 recorded years), and published evaluations put his accuracy well below a coin flip. Following Phil would lead to almost never buying seeds.
* **Farmer intuition**: looking at how the winter has gone (snow cover, warm spells).
* **Naive strategies** that our model must beat (computed in the cell below):
  * **Never buy**: profit = 0.
  * **Always buy**: profitable whenever early springs happen more than 20% of the time, which has been true since about the 2010s.
* **Seasonal outlooks**: NOAA's Climate Prediction Center publishes 1–3 month temperature outlooks. These are for regions, not this farm, and don't directly answer "will soil be ≥ 50 °F by May 1?"

## 4. How should you frame this problem?

* **Supervised learning**: we have ~80 years of history *with* the yearly early-spring label from the client.
* **Binary classification**: early spring (1) vs. not (0). One sample = one year.
* **Batch / offline learning**: data arrives once a year; no need for online learning.
* **Very small dataset**: ~80 rows minus missing years. This favors simple, well-regularized models, few but meaningful features, and repeated cross-validation (scores will be noisy).
* **Non-stationary**: the base rate has climbed from ~10–20% (pre-1990) to ~50% (2010s–2020s) because of climate change. The split and features must respect time (`year` is a legitimate feature for the trend).
* **Cost-sensitive**: errors are not equally bad (step 5), so we want calibrated probabilities plus a tuned decision threshold, not just a hard label.
* *Alternative framing to keep in mind:* regression on "how many days before/after May 1 the conditions are first met" (gives the model more information than yes/no), then threshold it. We start with classification because that is what the client needs.

In [1]:
# Expected profit per year (in units of R) of naive strategies, for different early-spring base rates
import pandas as pd

SEED_COST = 0.2           # seed cost as a fraction of the extra harvest's revenue (client)
GAIN = 1 - SEED_COST      # net gain when we buy and spring is early
PHIL_EARLY_RATE = 20/129  # Phil calls an early spring in ~15% of recorded years

def always_buy(p):    return p * GAIN - (1 - p) * SEED_COST
def never_buy(p):     return 0.0
def perfect(p):       return p * GAIN
def follow_phil(p):   return PHIL_EARLY_RATE * always_buy(p)   # if Phil is uninformative

baselines = pd.DataFrame(
    [{"early-spring rate": p,
      "never buy": never_buy(p),
      "always buy": always_buy(p),
      "follow Phil (no skill)": follow_phil(p),
      "perfect model": perfect(p),
      "headroom over best naive": perfect(p) - max(never_buy(p), always_buy(p))}
     for p in [0.10, 0.20, 0.35, 0.50, 0.60]]
).set_index("early-spring rate").round(3)

print(f"Break-even probability for buying seeds: {SEED_COST:.0%}")
print(f"Break-even precision (fraction of purchases that must pay off): {SEED_COST:.0%}  -> right 1 time out of 5 buys")
baselines

Break-even probability for buying seeds: 20%
Break-even precision (fraction of purchases that must pay off): 20%  -> right 1 time out of 5 buys


,never buy,always buy,follow Phil (no skill),perfect model,headroom over best naive
early-spring rate,,,,,
0.10,0.0,-0.10,-0.016,0.08,0.08
0.20,0.0,0.00,0.000,0.16,0.16
0.35,0.0,0.15,0.023,0.28,0.13
0.50,0.0,0.30,0.047,0.40,0.10
0.60,0.0,0.40,0.062,0.48,0.08


## 6. Is the performance measure aligned with the business objective?

Yes. Expected profit per year is the business objective, given the client's cost figure. Caveats to confirm with the client (step 12):

* It assumes the farm is risk-neutral, which means it can absorb several wasted seed purchases in a row.
* It assumes 1/5 is relative to the net revenue of the second harvest and that unused seeds have no salvage value.
* It assumes the decision is all-or-nothing. If the farm can buy seeds for only part of its fields, a calibrated probability becomes even more valuable.

*Note on the meeting notes:* we wrote "wrong 4/5 times and right 2/5 → break even". With the payoff table above, break-even is actually **1 correct purchase for every 4 wasted ones** (precision = 20%). We'll double-check this with the client.

## 7. What would be the minimum performance needed to reach the business objective?

The model is only worth using if it beats the  best naive strategy on recent years:

* Today (early-spring rate ≈ 50%): always buy earns +0.30 R/year, a perfect model earns +0.40 R/year. The model must earn*more than 0.30 R/year on the test years. The headroom is only ~0.10 R/year, so the main value is in skipping the clearly late springs.
* Before 1990 (rate ≈ 10–20%): *never buy* was optimal (0 R/year), and the model would have needed to find the rare early springs.

**Minimum bar:** test-set profit > max(always buy*, *never buy) over the same test years.

## 8. What are comparable problems? Can you reuse experience or tools?

* **Agricultural planting-date and last-frost forecasting**, e.g. predicting when soil reaches planting temperature.
* **Phenology models**: predicting first-bloom or leaf-out dates from winter temperatures (growing degree days).
* **Seasonal climate forecasting**: using teleconnections such as ENSO (El Niño/La Niña) and the Arctic Oscillation as predictors.
* **Cost-sensitive classification**, as in credit default or fraud detection: custom scorers, class weights, threshold tuning (`TunedThresholdClassifierCV`), and probability calibration.
* Reusable tools: scikit-learn pipelines, `RandomizedSearchCV`, repeated stratified CV for tiny datasets.

## 9. Is human expertise available?

* **The client**: knows the agronomy and the exact planting rules, and has the historical labels, owns the farm's weather/soil data.
* **Penn State Extension / agronomists**: soil temperature and planting knowledge for PA.
* **NOAA Climate Prediction Center / National Weather Service (Pittsburgh)**: how ENSO and the Arctic Oscillation affect late-winter and spring temperatures in western PA.

## 10. How would you solve the problem manually?

At the end of February, a farmer would look at:

1. **Soil temperature at 3 inches** during February. Is it warmer than usual and trending upward?
2. **Snow cover**: is there still snow on the ground?
3. **How the winter went overall**: mild or harsh? Was it an El Niño winter (often milder in the northern US)?
4. **Recent years**: early springs are now common, so lean toward buying.

Since buying only needs a > 20% chance of an early spring to pay off, a sensible manual rule is: buy unless late February is clearly cold and snowy. Our model formalizes and calibrates this rule. In the Exploration notebook we test a one-feature manual rule as an extra baseline.

## 11. List the assumptions you (or others) have made so far

1. **Label definition** is as in step 1. Specifically:
   * the 10 days start on the planting date, which is any day between March 1 and May 1;
   * "min air temp 30 °F" means *no day's minimum* goes below 30 °F;
   * "avg air temp 40 °F" and "avg soil temp 50 °F" are averages over the 10 days;
   * soil temperature is measured in the top 3 inches.
2. **Decision time:** only data available by March 1 can be used (data through Feb 28/29; ONI and AO indices through January).
3. **Costs:** seed cost = 0.2 × net revenue of the second harvest; no salvage value; all-or-nothing purchase; risk-neutral farm.
4. **Data:** daily farm weather for ~80 years including soil temperature, air min/max, and snow; the client's yearly labels follow the rules above.
5. **Missing years are dropped**, never filled in (client's instruction).
6. **Climate trend continues**: recent years are the most representative of the future, so we test on the most recent years.
7. **Phil's prediction** is a feature (`True` = Phil predicts an early spring, i.e. no shadow). We expect it to carry little signal.
8. Large-scale climate patterns (El Niño/La Niña, Arctic Oscillation) are related to late-winter and spring conditions in western PA ("weather patterns last a year").

### Questions for the client

1. **Label:** Is the planting date any day between ~March 1 and May 1? Is "min air temp ≥ 30 °F" a daily minimum on every one of the 10 days? Are your historical labels computed with exactly these rules?
2. **Timing:** Is data through the end of February usable, or do you need the answer earlier than March 1?
3. **Costs:** Is 1/5 relative to the second harvest's gross or net revenue? Do unused seeds have any salvage value? Could you buy seeds for only part of the farm?
4. **Risk:** How many wasted purchases in a row could the farm absorb?
5. **Data:** Format, columns and frequency (daily?), units, and depth of soil readings; which years are missing and how missing values are marked; approximate location (county) of the farm.
6. **Output:** Do you want a yes/no or a probability?

# Get the Data

## 1. List the data you need and how much you need

| Data | Why | Amount |
|---|---|---|
| Farm daily weather: soil temp (3 in), air min/max/mean, snow depth | Labels and the strongest features (late-winter soil temperature) | ~80 years × 365 days ≈ 29k rows |
| Farm yearly early-spring labels | Target | ~80 rows |
| Punxsutawney Phil's yearly prediction | Required input of `generate_samples()`; also a (weak) feature | 1886–present |
| **Oceanic Niño Index (ONI)**, NOAA CPC | External source #1: El Niño/La Niña state of the preceding fall/winter | 1950–present, monthly |
| **Arctic Oscillation (AO) index**, NOAA CPC | External source #2: strongly linked to cold outbreaks in the eastern US in winter | 1950–present, monthly |

We can't get more years than the farm has recorded, so **one row per year (~70–80 samples)** is a hard limit. Feature quality matters more than model complexity.

## 2. Find and document where you can get that data

| Source | Location |
|---|---|
| Farm data and labels | Jeff (client). **Email sent with the data request** — update `FARM_DATA_URL` / `FARM_LABELS_URL` below when received. |
| ONI | https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt |
| AO | https://www.cpc.ncep.noaa.gov/products/precip/CWlink/daily_ao_index/monthly.ao.index.b50.current.ascii |
| Phil's predictions | Wikipedia "Punxsutawney Phil" predictions table (compiled from groundhog.org / Stormfax), fetched via the Wikipedia API |

Possible extra sources if needed later: NOAA GHCN-Daily station data near the farm (to cross-check or gap-check the farm's sensors), NOAA CPC 90-day temperature outlooks.

## 3. Check how much space it will take

Tiny: the climate indices are ~50 KB each, the Wikipedia page ~200 KB, and the farm data probably < 5 MB. Everything fits in memory easily.

## 4. Check legal obligations, and get authorization if necessary

* NOAA data is produced by the U.S. government, so it is public domain.
* Wikipedia content is CC BY-SA, so we attribute it.
* Farm data is the client's private data. We use it only for this project and never commit it to the repository (also a project requirement). Only the derived `train_set.parquet` / `test_set.parquet` are committed.

## 5. Get access authorizations

The public sources need no authorization. Farm data: requested from Jeff.

## 6. Create a workspace

This repository. Raw downloads are cached in `data_cache/`, which is added to `.gitignore` (cell below).

## 7. Get the data

Code below. Every source is downloaded once and cached, so re-running the notebook doesn't hit the servers again.

In [2]:
import io
import json
import re
from collections import Counter
from functools import cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests   # if missing: pip install requests

CACHE_DIR = Path("data_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Make sure cached raw data never gets committed
gitignore = Path(".gitignore")
ignored = gitignore.read_text().splitlines() if gitignore.exists() else []
if "data_cache/" not in ignored:
    gitignore.write_text("\n".join(ignored + ["data_cache/"]) + "\n")


def fetch_cached(url: str, filename: str, refresh: bool = False) -> bytes:
    # Download `url` once into data_cache/filename and return its bytes.
    path = CACHE_DIR / filename
    if refresh or not path.exists():
        headers = {"User-Agent": "early-spring-class-project/1.0 (educational use)"}
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()   # clear error if the server refuses (e.g. 403/404)
        path.write_bytes(resp.content)
    return path.read_bytes()

### External source #1: Oceanic Niño Index (ONI)

Each row is a 3-month season, e.g. `NDJ 2025` = Nov 2025 – Jan 2026 (the `YR` column is the year of the *middle* month). For spring of year Y we can only use seasons that end by January of Y, since NOAA publishes each value about a week after the season ends. `DJF Y` would only be released in early March, which is after the order deadline, so it is not used.

In [3]:
ONI_URL = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

@cache
def load_oni() -> pd.DataFrame:
    raw = fetch_cached(ONI_URL, "oni.ascii.txt").decode()
    oni = pd.read_csv(io.StringIO(raw), sep=r"\s+")
    oni.columns = ["season", "year", "total", "anom"]
    return oni


def oni_features(years: list[int]) -> pd.DataFrame:
    # ONI anomalies known before March 1 of each year (all from seasons labelled year-1)
    piv = load_oni().pivot(index="year", columns="season", values="anom")
    get = lambda season, y: piv[season].get(y, np.nan)
    df = pd.DataFrame(index=pd.Index(years, name="year"))
    df["oni_son"] = [get("SON", y - 1) for y in years]  # Sep-Nov
    df["oni_ond"] = [get("OND", y - 1) for y in years]  # Oct-Dec
    df["oni_ndj"] = [get("NDJ", y - 1) for y in years]  # Nov-Jan (latest available by March 1)
    df["oni_trend"] = df["oni_ndj"] - [get("JJA", y - 1) for y in years]  # warming/cooling since summer
    return df

### External source #2: Arctic Oscillation (AO) index

Monthly values (`year month value`). A negative AO in winter usually means cold air outbreaks into the eastern US. For spring of year Y we use Nov and Dec of Y−1 and Jan of Y (February's value is only final after February ends).

In [4]:
AO_URL = "https://www.cpc.ncep.noaa.gov/products/precip/CWlink/daily_ao_index/monthly.ao.index.b50.current.ascii"

@cache
def load_ao() -> pd.Series:
    raw = fetch_cached(AO_URL, "ao_monthly.ascii").decode()
    ao = pd.read_csv(io.StringIO(raw), sep=r"\s+", header=None, names=["year", "month", "ao"])
    return ao.set_index(["year", "month"])["ao"]


def ao_features(years: list[int]) -> pd.DataFrame:
    ao = load_ao()
    get = lambda y, m: ao.get((y, m), np.nan)
    df = pd.DataFrame(index=pd.Index(years, name="year"))
    df["ao_nov"] = [get(y - 1, 11) for y in years]
    df["ao_dec"] = [get(y - 1, 12) for y in years]
    df["ao_jan"] = [get(y, 1) for y in years]
    df["ao_winter_mean"] = df[["ao_nov", "ao_dec", "ao_jan"]].mean(axis=1, skipna=False)
    return df

### Farm data from Jeffery

Four CSV files. They are the client's private data, so they are **not committed**:

* If Jeffery's files are online, put their links in `FARM_URLS` and the code downloads and caches them, like the other sources.
* Otherwise, copy the four files into `data_cache/` (git-ignored) with the names below.

Things found while reading them:

* `weather.csv` marks missing values as **`M`** (→ NaN) and trace precipitation/snow as **`T`** (→ 0.0).
* `soil.csv` has no missing values. Soil temperature looks like **°C** (≈ 0 in January; the 50 °F planting rule = 10 °C).
* Both daily files cover **Jan 1 – Mar 1** of each year. March 1 is excluded from the features (see Frame the Problem, assumption 2).

In [5]:
FARM_FILES = {            
    "soil": "soil.csv",
    "weather": "weather.csv",
    "labels": "early_spring.csv",
    "phil": "phil_pred.csv",
}
FARM_URLS = {              # TODO (optional): links to Jeffery's files, if they are online
    "soil": None, "weather": None, "labels": None, "phil": None,
}


def farm_file(key: str) -> Path:
    name = FARM_FILES[key]
    if FARM_URLS.get(key):
        fetch_cached(FARM_URLS[key], name)
    # look in data_cache/ first, then in the project root next to the notebook
    for path in (CACHE_DIR / name, Path(name)):
        if path.exists():
            return path
    raise FileNotFoundError(f"{name} not found in {CACHE_DIR}/ or in {Path.cwd()}")


@cache
def load_farm_daily() -> pd.DataFrame:
    soil = pd.read_csv(farm_file("soil"), parse_dates=["day"]).rename(columns={"day": "date"})
    weather = pd.read_csv(farm_file("weather"), parse_dates=["date"], dtype=str, keep_default_na=False)
    weather["date"] = pd.to_datetime(weather["date"])
    for col in weather.columns.drop("date"):
        weather[col] = pd.to_numeric(weather[col].str.strip().replace({"T": "0", "M": "", "": None}), errors="coerce")
    df = soil.merge(weather, on="date", how="outer").set_index("date").sort_index()
    # avg_temp is missing more often than min/max; derive it from them when possible (same day, not filled from other days)
    df["avg_temp"] = df["avg_temp"].fillna((df["max_temp"] + df["min_temp"]) / 2)
    return df


@cache
def load_labels() -> pd.Series:
    labels = pd.read_csv(farm_file("labels"))
    return labels.set_index("year")["early_spring"].astype("boolean").sort_index()


@cache
def load_phil() -> pd.Series:
    # True = Phil predicted an early spring
    phil = pd.read_csv(farm_file("phil"))
    return phil.set_index("year")["prediction"].astype("boolean").sort_index()

In [6]:
LATE_FEB_START = 15   # "late February" = Feb 15 to Feb 28/29


def _n(s: pd.Series, mask: pd.Series) -> float:
    # number of days where mask is True; NaN if the whole window has no readings (never pretend it's "0 days")
    return float(mask.sum()) if s.notna().any() else np.nan


def farm_features(years: list[int]) -> pd.DataFrame:
    # Farm features from Jan 1 to the last day of Feb (March 1 excluded).
    daily = load_farm_daily()
    rows = {}
    for y in years:
        group = daily[(daily.index >= pd.Timestamp(y, 1, 1)) & (daily.index < pd.Timestamp(y, 3, 1))]
        if group[["mean_soil_temp", "avg_temp", "min_temp"]].dropna(how="all").empty:
            raise ValueError(f"No farm data for {y} before March 1 (missing years are dropped, never filled in).")
        jan = group[group.index.month == 1]
        feb = group[group.index.month == 2]
        feb_late = feb[feb.index.day >= LATE_FEB_START]

        rows[y] = {
            # --- the features you asked for ---
            "snow_days":     _n(feb["snowdepth"], feb["snowdepth"] > 0),
            "cold_days":     _n(feb["min_temp"], feb["min_temp"] < 20.0),
            "gdd":           np.maximum(0, group["avg_temp"] - 40.0).sum(),   # growing degree days, Jan 1 - Feb end
            "air_feb_late":  feb_late["avg_temp"].mean(),
            "soil_feb_late": feb_late["mean_soil_temp"].mean(),
            "soil_moisture": feb["mean_soil_moisture"].mean(),
            # --- extras of the same kind ---
            "air_feb":            feb["avg_temp"].mean(),
            "soil_feb":           feb["mean_soil_temp"].mean(),
            "soil_trend":         feb_late["mean_soil_temp"].mean() - jan["mean_soil_temp"].mean(),  # warming since January
            "snow_days_late":     _n(feb_late["snowdepth"], feb_late["snowdepth"] > 0),
            "soil_moisture_late": feb_late["mean_soil_moisture"].mean(),
        }
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("year")

In [7]:
def generate_samples(years: list[int], phil_preds: list[bool]) -> pd.DataFrame:
    # One row per year: Phil's call + farm features + ONI + AO. No label column.
    if len(years) != len(phil_preds):
        raise ValueError(f"years ({len(years)}) and phil_preds ({len(phil_preds)}) must have the same length")
    years = [int(y) for y in years]
    X = pd.concat([farm_features(years), oni_features(years), ao_features(years)], axis=1)
    X.insert(0, "phil_early", pd.array([bool(p) for p in phil_preds], dtype="boolean"))
    return X.reset_index()          # `year` becomes the first column

In [8]:
N_TEST_YEARS = 15
TARGET = "early_spring"

labels, phil = load_labels(), load_phil()
farm_years = set(load_farm_daily().index.year)

candidates = sorted(set(labels.index) | set(phil.index))
keep, dropped = [], {}
for y in candidates:
    if pd.isna(labels.get(y, pd.NA)):  dropped[y] = "no label"
    elif pd.isna(phil.get(y, pd.NA)):  dropped[y] = "no Phil prediction"
    elif y not in farm_years:          dropped[y] = "no farm weather/soil data"
    else:                              keep.append(y)

# also drop years where the whole Jan-Feb window has no usable values
usable = []
for y in keep:
    try:
        farm_features([y]); usable.append(y)
    except ValueError:
        dropped[y] = "no usable farm data before March 1"

print(f"Kept {len(usable)} years ({usable[0]}-{usable[-1]}); dropped {len(dropped)}: {dict(sorted(dropped.items()))}")

data = generate_samples(usable, phil.loc[usable].tolist())
data[TARGET] = labels.loc[usable].astype(bool).to_numpy()

train_set = data[data["year"] <= usable[-N_TEST_YEARS - 1]].reset_index(drop=True)
test_set  = data[data["year"] >  usable[-N_TEST_YEARS - 1]].reset_index(drop=True)

print(f"train: {len(train_set)} years ({train_set.year.min()}-{train_set.year.max()}), early-spring rate {train_set[TARGET].mean():.0%}")
print(f"test:  {len(test_set)} years ({test_set.year.min()}-{test_set.year.max()}), early-spring rate {test_set[TARGET].mean():.0%}")
train_set.head()

Kept 72 years (1947-2026); dropped 0: {}
train: 57 years (1947-2010), early-spring rate 19%
test:  15 years (2012-2026), early-spring rate 47%


,year,phil_early,snow_days,cold_days,gdd,air_feb_late,soil_feb_late,soil_moisture,air_feb,soil_feb,...,soil_moisture_late,oni_son,oni_ond,oni_ndj,oni_trend,ao_nov,ao_dec,ao_jan,ao_winter_mean,early_spring
0,1947,False,19.0,21.0,58.0,23.571429,-0.052679,0.464049,21.642857,-1.873661,...,0.464661,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1948,False,14.0,15.0,31.5,36.966667,1.385278,0.471476,28.586207,0.754023,...,0.492517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1949,False,3.0,11.0,115.5,40.535714,3.111905,0.485908,36.750000,2.362649,...,0.487057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
3,1950,True,0.0,10.0,136.5,27.678571,-0.453274,0.476948,30.803571,0.302827,...,0.477164,NaN,NaN,NaN,NaN,NaN,NaN,-0.0603,NaN,False
4,1951,False,4.0,8.0,62.5,39.071429,2.026190,0.483156,32.214286,0.772024,...,0.479914,-0.62,-0.66,-0.79,-0.17,-0.5151,-1.9281,-0.0850,-0.842733,True


In [10]:
%pip install pyarrow
train_set.to_parquet("train_set.parquet", index=False)
test_set.to_parquet("test_set.parquet", index=False)

# --- sanity checks ---
for name in ["train_set", "test_set"]:
    df = pd.read_parquet(f"{name}.parquet")
    feats = df.drop(columns=TARGET)
    again = generate_samples(df["year"].tolist(), df["phil_early"].tolist())
    pd.testing.assert_frame_equal(feats, again, check_dtype=False)   # generate_samples reproduces the file exactly
    print(f"{name}.parquet OK: {df.shape[0]} rows x {df.shape[1]} cols, "
          f"{int(feats.drop(columns='year').isna().any(axis=1).sum())} rows with at least one NaN feature")

print("\nFeature columns:", [c for c in train_set.columns if c not in ("year", TARGET)])

  Using cached pyarrow-26.0.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (3.0 kB)
Using cached pyarrow-26.0.0-cp314-cp314-macosx_12_0_arm64.whl (36.4 MB)

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
train_set.parquet OK: 57 rows x 22 cols, 5 rows with at least one NaN feature
test_set.parquet OK: 15 rows x 22 cols, 0 rows with at least one NaN feature

Feature columns: ['phil_early', 'snow_days', 'cold_days', 'gdd', 'air_feb_late', 'soil_feb_late', 'soil_moisture', 'air_feb', 'soil_feb', 'soil_trend', 'snow_days_late', 'soil_moisture_late', 'oni_son', 'oni_ond', 'oni_ndj', 'oni_trend', 'ao_nov', 'ao_dec', 'ao_jan', 'ao_winter_mean']
